# TexPilot — manually run Kaggle baseline

Prepared locally; **no cells have been executed on Kaggle by the agent**.

You operate Kaggle yourself. Start with Accelerator set to None and run only the first input-inventory cell. Attach the actual TextileNet image snapshot **with its frozen `splits/fabric.csv`** before enabling GPU and Internet and continuing through the remaining cells. No Kaggle API keys or credentials are used here.

This notebook runs a frozen DINOv2 fabric-structure baseline and selects hyperparameters using validation top-1, matching the repository baseline objective. It also reports macro F1, weighted F1, micro F1, balanced accuracy and per-class results. Runtime dependency versions are recorded; do not call this an exact historical reproduction unless the data snapshot and environment match.

The notebook refuses missing data and exact cross-split hash leakage. Its audit uses manifest hashes; it is not a fresh image-integrity or near-duplicate audit. No dataset is silently substituted. The held-out test split is excluded from feature extraction and selection; its optional final-evaluation cell is disabled by default.

Download `texpilot_results.zip` after a run. The archive contains provenance, logs, full validation metrics, the frozen split, and the selected linear head plus normalization parameters. Feature caches remain in `/kaggle/working` for resumption within the session.


In [ ]:
# START HERE: inspect attached inputs without starting a GPU session.
from pathlib import Path
import shutil

inputs = Path('/kaggle/input')
folders = sorted(inputs.iterdir()) if inputs.exists() else []
print('Attached input folders:')
for folder in folders:
    print(' ', folder)
if not folders:
    print('  NONE — attach the actual TextileNet snapshot before enabling a GPU.')
candidates = sorted(inputs.glob('**/splits/fabric.csv')) if inputs.exists() else []
print('Frozen fabric split files:', [str(path) for path in candidates])
if not candidates:
    print('No frozen fabric.csv found. The notebook is not ready for training yet.')
print('Free working disk GiB:', round(shutil.disk_usage('/kaggle/working').free / 2**30, 2))


In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys, time
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is visible. Enable a Kaggle GPU before continuing.')
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Visible GPUs:', torch.cuda.device_count(), '(this notebook uses cuda:0 only)')
# Confirm CUDA kernels and backward work, not just that a GPU is visible.
probe_tensor = torch.ones((32, 32), device='cuda', requires_grad=True)
(probe_tensor.square().sum()).backward()
torch.cuda.synchronize()
del probe_tensor
print('CUDA forward/backward smoke check: passed')
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('Free working disk GiB:', round(shutil.disk_usage('/kaggle/working').free / 2**30, 2))
subprocess.run(['nvidia-smi'], check=True)
print('Kaggle quota and session duration must be checked in the UI by you.')


In [ ]:
# Preserve Kaggle's existing torch/torchvision/NumPy versions during dependency resolution.
# The exact resolved environment is saved later; it is not the historical pinned environment.
from importlib.metadata import version
constraints = Path('/kaggle/working/texpilot_runtime_constraints.txt')
constraints.write_text(''.join(f'{package}=={version(package)}\n' for package in ['torch', 'torchvision', 'numpy']))
print(constraints.read_text())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(constraints),
                'timm>=1.0,<1.1', 'albumentations>=2.0,<2.1', 'matplotlib'], check=True)


In [ ]:
import base64, hashlib
REPO = Path('/kaggle/working/TexPilot')
BASE_COMMIT = 'd6fa46b216b25144b0cd8388e53a94139a871a42'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/AnasTabba/TexPilot.git', str(REPO)], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if actual != BASE_COMMIT:
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', BASE_COMMIT], check=True)
OVERLAY = {'training/textilenet/metrics.py': 'IiIiQ2xhc3NpZmljYXRpb24gbWV0cmljcyBmcm9tIGxvZ2l0cy4gbnVtcHkgb25seTsgc2hhcmVkIGJ5IHRyYWluLCBwcm9iZSBhbmQgYWdncmVnYXRlLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCgoKZGVmIHRvcGtfYWNjdXJhY3kobG9naXRzOiBucC5uZGFycmF5LCB5OiBucC5uZGFycmF5LCBrOiBpbnQpIC0+IGZsb2F0OgogICAgayA9IG1pbihrLCBsb2dpdHMuc2hhcGVbMV0pCiAgICB0b3BrID0gbnAuYXJncGFydGl0aW9uKC1sb2dpdHMsIGsgLSAxLCBheGlzPTEpWzosIDprXQogICAgcmV0dXJuIGZsb2F0KCh0b3BrID09IHlbOiwgTm9uZV0pLmFueShheGlzPTEpLm1lYW4oKSkKCgpkZWYgY29uZnVzaW9uX21hdHJpeCh5OiBucC5uZGFycmF5LCBwcmVkOiBucC5uZGFycmF5LCBuX2NsYXNzZXM6IGludCkgLT4gbnAubmRhcnJheToKICAgICIiIlJvd3MgYXJlIHRydWUgY2xhc3NlcywgY29sdW1ucyBwcmVkaWN0aW9ucy4iIiIKICAgIHJldHVybiBucC5iaW5jb3VudCh5ICogbl9jbGFzc2VzICsgcHJlZCwgbWlubGVuZ3RoPW5fY2xhc3NlcyoqMikucmVzaGFwZShuX2NsYXNzZXMsIG5fY2xhc3NlcykKCgpkZWYgc3VtbWFyaXplKGxvZ2l0czogbnAubmRhcnJheSwgeTogbnAubmRhcnJheSwgY2xhc3NlczogbGlzdFtzdHJdLCBuX3BhaXJzOiBpbnQgPSAxNSkgLT4gZGljdDoKICAgICIiIlRvcC0xLzUsIG1hY3JvLUYxLCBwZXItY2xhc3MgcmVjYWxsL3ByZWNpc2lvbi9GMSwgYW5kIHRoZSBtb3N0LWNvbmZ1c2VkIHBhaXJzLgoKICAgIE1hY3JvLUYxIGF2ZXJhZ2VzIG92ZXIgY2xhc3NlcyBwcmVzZW50IGluIGBgeWBgOiBhIGNsYXNzIGFic2VudCBmcm9tIHRoZSBldmFsdWF0ZWQKICAgIHNldCAocG9zc2libGUgYWZ0ZXIgbGluayByb3QpIGhhcyBubyBkZWZpbmVkIHJlY2FsbCBhbmQgaXMgcmVwb3J0ZWQsIG5vdCBhdmVyYWdlZC4KICAgICIiIgogICAgbiA9IGxlbihjbGFzc2VzKQogICAgcHJlZCA9IGxvZ2l0cy5hcmdtYXgoMSkKICAgIGNtID0gY29uZnVzaW9uX21hdHJpeCh5LCBwcmVkLCBuKQogICAgdHAgPSBucC5kaWFnKGNtKS5hc3R5cGUoZmxvYXQpCiAgICBzdXBwb3J0ID0gY20uc3VtKDEpCiAgICBwcmVkaWN0ZWQgPSBjbS5zdW0oMCkKICAgIHdpdGggbnAuZXJyc3RhdGUoZGl2aWRlPSJpZ25vcmUiLCBpbnZhbGlkPSJpZ25vcmUiKToKICAgICAgICByZWNhbGwgPSBucC53aGVyZShzdXBwb3J0ID4gMCwgdHAgLyBzdXBwb3J0LCBucC5uYW4pCiAgICAgICAgcHJlY2lzaW9uID0gbnAud2hlcmUocHJlZGljdGVkID4gMCwgdHAgLyBwcmVkaWN0ZWQsIDAuMCkKICAgICAgICBmMSA9IG5wLndoZXJlKAogICAgICAgICAgICAocHJlY2lzaW9uICsgbnAubmFuX3RvX251bShyZWNhbGwpKSA+IDAsCiAgICAgICAgICAgIDIgKiBwcmVjaXNpb24gKiByZWNhbGwgLyAocHJlY2lzaW9uICsgcmVjYWxsKSwKICAgICAgICAgICAgMC4wLAogICAgICAgICkKICAgIHByZXNlbnQgPSBzdXBwb3J0ID4gMAogICAgZXZhbHVhdGVkID0gc3VwcG9ydC5zdW0oKQoKICAgIHJhdGVzID0gbnAud2hlcmUoc3VwcG9ydFs6LCBOb25lXSA+IDAsIGNtIC8gbnAubWF4aW11bShzdXBwb3J0WzosIE5vbmVdLCAxKSwgMC4wKQogICAgbnAuZmlsbF9kaWFnb25hbChyYXRlcywgMCkKICAgIHBhaXJzID0gW10KICAgIGZvciBmbGF0IGluIG5wLmFyZ3NvcnQoLXJhdGVzLCBheGlzPU5vbmUpWzpuX3BhaXJzXToKICAgICAgICBpLCBqID0gZGl2bW9kKGludChmbGF0KSwgbikKICAgICAgICBpZiBjbVtpLCBqXSA9PSAwOgogICAgICAgICAgICBicmVhawogICAgICAgIHBhaXJzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgInRydWUiOiBjbGFzc2VzW2ldLAogICAgICAgICAgICAgICAgInByZWQiOiBjbGFzc2VzW2pdLAogICAgICAgICAgICAgICAgImNvdW50IjogaW50KGNtW2ksIGpdKSwKICAgICAgICAgICAgICAgICJyYXRlIjogcm91bmQoZmxvYXQocmF0ZXNbaSwgal0pLCA0KSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICByZXR1cm4gewogICAgICAgICJuIjogaW50KGxlbih5KSksCiAgICAgICAgInRvcDEiOiB0b3BrX2FjY3VyYWN5KGxvZ2l0cywgeSwgMSksCiAgICAgICAgInRvcDUiOiB0b3BrX2FjY3VyYWN5KGxvZ2l0cywgeSwgNSksCiAgICAgICAgIm1hY3JvX2YxIjogZmxvYXQobnAubmFubWVhbihucC53aGVyZShwcmVzZW50LCBmMSwgbnAubmFuKSkpLAogICAgICAgICJ3ZWlnaHRlZF9mMSI6IGZsb2F0KG5wLm5hbnN1bShmMSAqIHN1cHBvcnQpIC8gZXZhbHVhdGVkKSBpZiBldmFsdWF0ZWQgZWxzZSBOb25lLAogICAgICAgICJtaWNyb19mMSI6IGZsb2F0KHRwLnN1bSgpIC8gZXZhbHVhdGVkKSBpZiBldmFsdWF0ZWQgZWxzZSBOb25lLAogICAgICAgICJiYWxhbmNlZF9hY2N1cmFjeSI6IGZsb2F0KG5wLm5hbm1lYW4obnAud2hlcmUocHJlc2VudCwgcmVjYWxsLCBucC5uYW4pKSkKICAgICAgICBpZiBldmFsdWF0ZWQKICAgICAgICBlbHNlIE5vbmUsCiAgICAgICAgImNsYXNzZXNfYWJzZW50IjogW2MgZm9yIGMsIHAgaW4gemlwKGNsYXNzZXMsIHByZXNlbnQsIHN0cmljdD1UcnVlKSBpZiBub3QgcF0sCiAgICAgICAgInBlcl9jbGFzcyI6IHsKICAgICAgICAgICAgYzogewogICAgICAgICAgICAgICAgInN1cHBvcnQiOiBpbnQoc3VwcG9ydFtpXSksCiAgICAgICAgICAgICAgICAicmVjYWxsIjogX2YocmVjYWxsW2ldKSwKICAgICAgICAgICAgICAgICJwcmVjaXNpb24iOiBfZihwcmVjaXNpb25baV0pLAogICAgICAgICAgICAgICAgImYxIjogX2YoZjFbaV0pLAogICAgICAgICAgICB9CiAgICAgICAgICAgIGZvciBpLCBjIGluIGVudW1lcmF0ZShjbGFzc2VzKQogICAgICAgIH0sCiAgICAgICAgImNvbmZ1c2VkX3BhaXJzIjogcGFpcnMsCiAgICAgICAgImNvbmZ1c2lvbl9tYXRyaXgiOiBjbS50b2xpc3QoKSwKICAgIH0KCgpkZWYgX2YoeDogZmxvYXQpIC0+IGZsb2F0IHwgTm9uZToKICAgIHJldHVybiBOb25lIGlmIG5wLmlzbmFuKHgpIGVsc2Ugcm91bmQoZmxvYXQoeCksIDQpCg==', 'training/textilenet/audit.py': 'IiIiUmVwcm9kdWNpYmxlIGNsYXNzLWZyZXF1ZW5jeSBhbmQgZXhhY3QtbGVha2FnZSBhdWRpdCBmb3IgYSBmcm96ZW4gVGV4dGlsZU5ldCBzcGxpdC4KClVzYWdlOiBweXRob24gLW0gdHJhaW5pbmcudGV4dGlsZW5ldC5hdWRpdCAtLXBhcnRpdGlvbiBmYWJyaWMgXAogICAgLS1zcGxpdC1jc3YgZGF0YS9zcGxpdHMvZmFicmljLmNzdiAtLWRhdGEtcm9vdCBkYXRhIC0tb3V0IHJ1bnMvYXVkaXQvZmFicmljLmpzb24KClRoaXMgZG9lcyBub3QgaW5mZXIgbmVhci1kdXBsaWNhdGVzLCBsYWJlbHMgb3IgaW1hZ2UgcXVhbGl0eSBmcm9tIGZpbGVuYW1lcy4gVGhvc2UKcmVxdWlyZSBhIHNlcGFyYXRlIGltYWdlLWxldmVsIHJldmlldy4gSXQgbmV2ZXIgbW9kaWZpZXMgdGhlIGZyb3plbiBzcGxpdC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlciwgZGVmYXVsdGRpY3QKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIHRyYWluaW5nLnRleHRpbGVuZXQuc3BsaXRzIGltcG9ydCBQQVJUSVRJT05TLCBSZWNvcmQsIHJlYWRfY3N2CgoKZGVmIF9zcGxpdF9zdW1tYXJ5KHJvd3M6IGxpc3RbUmVjb3JkXSwgY2xhc3NlczogdHVwbGVbc3RyLCAuLi5dKSAtPiBkaWN0OgogICAgY291bnRzID0gQ291bnRlcihyb3cubGFiZWwgZm9yIHJvdyBpbiByb3dzKQogICAgdmFsdWVzID0gW2NvdW50c1tsYWJlbF0gZm9yIGxhYmVsIGluIGNsYXNzZXNdCiAgICBwcmVzZW50ID0gW3ZhbHVlIGZvciB2YWx1ZSBpbiB2YWx1ZXMgaWYgdmFsdWVdCiAgICBuID0gc3VtKHZhbHVlcykKICAgIGVudHJvcHkgPSAtc3VtKCh2YWx1ZSAvIG4pICogbWF0aC5sb2codmFsdWUgLyBuKSBmb3IgdmFsdWUgaW4gcHJlc2VudCkgaWYgbiBlbHNlIE5vbmUKICAgIHJldHVybiB7CiAgICAgICAgIm4iOiBuLAogICAgICAgICJjbGFzc19jb3VudHMiOiB7bGFiZWw6IGNvdW50c1tsYWJlbF0gZm9yIGxhYmVsIGluIGNsYXNzZXN9LAogICAgICAgICJhYnNlbnRfY2xhc3NlcyI6IFtsYWJlbCBmb3IgbGFiZWwgaW4gY2xhc3NlcyBpZiBub3QgY291bnRzW2xhYmVsXV0sCiAgICAgICAgIm1ham9yaXR5X21pbm9yaXR5X3JhdGlvIjogbWF4KHByZXNlbnQpIC8gbWluKHByZXNlbnQpIGlmIHByZXNlbnQgZWxzZSBOb25lLAogICAgICAgICJlbnRyb3B5X25vcm1hbGl6ZWQiOiBlbnRyb3B5IC8gbWF0aC5sb2cobGVuKGNsYXNzZXMpKQogICAgICAgIGlmIGVudHJvcHkgaXMgbm90IE5vbmUgYW5kIGxlbihjbGFzc2VzKSA+IDEKICAgICAgICBlbHNlIE5vbmUsCiAgICB9CgoKZGVmIGF1ZGl0X3JlY29yZHMocm93czogbGlzdFtSZWNvcmRdLCBkYXRhX3Jvb3Q6IFBhdGggfCBOb25lLCBjbGFzc2VzOiB0dXBsZVtzdHIsIC4uLl0pIC0+IGRpY3Q6CiAgICAiIiJBdWRpdCBtYW5pZmVzdCBjb3VudHMgYW5kIGJ5dGUtaWRlbnRpY2FsIGNyb3NzLXNwbGl0IGhhc2hlcyB3aXRob3V0IHRvdWNoaW5nIGRhdGEuIiIiCiAgICB1bmtub3duID0gc29ydGVkKHtyb3cubGFiZWwgZm9yIHJvdyBpbiByb3dzfSAtIHNldChjbGFzc2VzKSkKICAgIGlmIHVua25vd246CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gbGFiZWxzOiB7dW5rbm93bn0iKQogICAgYnlfaGFzaDogZGljdFtzdHIsIHNldFtzdHJdXSA9IGRlZmF1bHRkaWN0KHNldCkKICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICBpZiByb3cuc2hhMToKICAgICAgICAgICAgYnlfaGFzaFtyb3cuc2hhMV0uYWRkKHJvdy5zcGxpdCkKICAgIHJldHVybiB7CiAgICAgICAgInRvdGFsX3Jvd3MiOiBsZW4ocm93cyksCiAgICAgICAgInNvdXJjZXMiOiBkaWN0KHNvcnRlZChDb3VudGVyKHJvdy5zb3VyY2UgZm9yIHJvdyBpbiByb3dzKS5pdGVtcygpKSksCiAgICAgICAgInNwbGl0cyI6IHsKICAgICAgICAgICAgc3BsaXQ6IF9zcGxpdF9zdW1tYXJ5KFtyb3cgZm9yIHJvdyBpbiByb3dzIGlmIHJvdy5zcGxpdCA9PSBzcGxpdF0sIGNsYXNzZXMpCiAgICAgICAgICAgIGZvciBzcGxpdCBpbiAoInRyYWluIiwgInZhbCIsICJ0ZXN0IikKICAgICAgICB9LAogICAgICAgICJtaXNzaW5nX2ZpbGVzIjogc29ydGVkKAogICAgICAgICAgICByb3cucGF0aAogICAgICAgICAgICBmb3Igcm93IGluIHJvd3MKICAgICAgICAgICAgaWYgZGF0YV9yb290IGlzIG5vdCBOb25lIGFuZCBub3QgKGRhdGFfcm9vdCAvIHJvdy5wYXRoKS5pc19maWxlKCkKICAgICAgICApLAogICAgICAgICJjcm9zc19zcGxpdF9zaGExIjogWwogICAgICAgICAgICB7InNoYTEiOiBzaGExLCAic3BsaXRzIjogc29ydGVkKHNwbGl0cyl9CiAgICAgICAgICAgIGZvciBzaGExLCBzcGxpdHMgaW4gc29ydGVkKGJ5X2hhc2guaXRlbXMoKSkKICAgICAgICAgICAgaWYgbGVuKHNwbGl0cykgPiAxCiAgICAgICAgXSwKICAgICAgICAibmVhcl9kdXBsaWNhdGVfc3RhdHVzIjogIm5vdF9tZWFzdXJlZCIsCiAgICB9CgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wYXJ0aXRpb24iLCBjaG9pY2VzPXR1cGxlKFBBUlRJVElPTlMpLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zcGxpdC1jc3YiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEtcm9vdCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCJkYXRhIikpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBpZiBub3QgYXJncy5zcGxpdF9jc3YuaXNfZmlsZSgpOgogICAgICAgIHBhcnNlci5lcnJvcihmImZyb3plbiBzcGxpdCBub3QgZm91bmQ6IHthcmdzLnNwbGl0X2Nzdn07IG9idGFpbiB0aGUgZGF0YXNldCBmaXJzdCIpCiAgICByZXBvcnQgPSBhdWRpdF9yZWNvcmRzKHJlYWRfY3N2KGFyZ3Muc3BsaXRfY3N2KSwgYXJncy5kYXRhX3Jvb3QsIFBBUlRJVElPTlNbYXJncy5wYXJ0aXRpb25dKQogICAgcmVwb3J0LnVwZGF0ZSh7InBhcnRpdGlvbiI6IGFyZ3MucGFydGl0aW9uLCAic3BsaXRfY3N2Ijogc3RyKGFyZ3Muc3BsaXRfY3N2KX0pCiAgICBhcmdzLm91dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgYXJncy5vdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCwgaW5kZW50PTIpICsgIlxuIikKICAgIHByaW50KAogICAgICAgIGYid3JvdGUge2FyZ3Mub3V0fToge3JlcG9ydFsndG90YWxfcm93cyddfSByb3dzLCAiCiAgICAgICAgZiJ7bGVuKHJlcG9ydFsnbWlzc2luZ19maWxlcyddKX0gbWlzc2luZyBmaWxlcyIKICAgICkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo='}
OVERLAY_SHA256 = {'training/textilenet/metrics.py': 'c57c5594d0a95770ff8f608a0ef9344d0fae05b4143adc5ebe841c0386477807', 'training/textilenet/audit.py': '944b7609dc31b467478f1af36f57274383f660a321ce59d8ba00d3c20cfa2c6d'}
for relative, encoded in OVERLAY.items():
    payload = base64.b64decode(encoded)
    assert hashlib.sha256(payload).hexdigest() == OVERLAY_SHA256[relative]
    target = REPO / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('Pinned repository:', BASE_COMMIT)
print('Local audit/metric overlay:', OVERLAY_SHA256)


## Locate the actual dataset

If exactly one attached dataset contains `splits/fabric.csv`, it is selected automatically. Otherwise set both paths below to the attached snapshot. Image paths in the frozen CSV must resolve relative to `DATA_ROOT`. Do not rebuild a split just to make a historical result appear reproducible.


In [ ]:
DATA_ROOT = None  # Example: Path('/kaggle/input/<your-attached-snapshot>')
SPLIT_CSV = None  # Example: DATA_ROOT / 'splits/fabric.csv'
DATASET_VERSION = ''  # Fill with the dataset's immutable version/provenance identifier.

if DATA_ROOT is None:
    candidates = sorted(Path('/kaggle/input').glob('**/splits/fabric.csv'))
    print('Frozen fabric split candidates:', candidates)
    if len(candidates) != 1:
        raise RuntimeError('Attach the TextileNet snapshot and explicitly set DATA_ROOT and SPLIT_CSV.')
    SPLIT_CSV = candidates[0]
    DATA_ROOT = SPLIT_CSV.parent.parent
DATA_ROOT = Path(DATA_ROOT)
SPLIT_CSV = Path(SPLIT_CSV or DATA_ROOT / 'splits/fabric.csv')
if not SPLIT_CSV.is_file():
    raise FileNotFoundError(f'Frozen split not found: {SPLIT_CSV}')
if not DATASET_VERSION.strip():
    raise ValueError('Set DATASET_VERSION to the exact attached snapshot identifier before training.')
print('Dataset:', DATA_ROOT, 'Split:', SPLIT_CSV, 'Version:', DATASET_VERSION)


In [ ]:
from training.textilenet.audit import audit_records
from training.textilenet.splits import PARTITIONS, read_csv

RESULTS = Path('/kaggle/working/texpilot_results')
RESULTS.mkdir(exist_ok=True)
rows = read_csv(SPLIT_CSV)
report = audit_records(rows, DATA_ROOT, PARTITIONS['fabric'])
(RESULTS / 'audit.json').write_text(json.dumps(report, indent=2))
print(json.dumps({key: report[key] for key in ['total_rows', 'sources', 'splits']}, indent=2))
if report['missing_files']:
    raise RuntimeError(f"{len(report['missing_files'])} missing images; see audit.json. No rows were silently dropped.")
if report['cross_split_sha1']:
    raise RuntimeError('Manifest hashes identify cross-split duplicates. Review the frozen split before training.')
if any(report['splits'][s]['n'] == 0 for s in ['train', 'val', 'test']):
    raise RuntimeError('A required split is empty.')
if report['splits']['train']['absent_classes'] or report['splits']['val']['absent_classes']:
    raise RuntimeError('Classes are absent from train or validation. Review coverage before comparison.')
shutil.copy2(SPLIT_CSV, RESULTS / 'fabric.csv')
provenance = {
    'base_commit': BASE_COMMIT, 'overlay_sha256': OVERLAY_SHA256,
    'dataset_version': DATASET_VERSION,
    'split_sha256': hashlib.sha256(SPLIT_CSV.read_bytes()).hexdigest(),
    'data_root': str(DATA_ROOT), 'split_csv': str(SPLIT_CSV),
    'gpu': torch.cuda.get_device_name(0), 'python': sys.version,
    'torch': torch.__version__, 'selection_metric': 'val_top1',
    'test_evaluated': False,
}
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))
(RESULTS / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


In [ ]:
from argparse import Namespace
import numpy as np
from training.textilenet.probe import extract, fit_linear, GRID
from training.textilenet.metrics import summarize
from training.textilenet.splits import class_index

SEED = 0
EPOCHS = 30
args = Namespace(partition='fabric', model='dinov2_vitb14', img_size=224,
                 crop_pct=0.875, data_root=DATA_ROOT,
                 feature_dir=Path('/kaggle/working/texpilot_features'), batch_size=32, workers=2)
device = torch.device('cuda')
labels = class_index('fabric')
classes = sorted(labels, key=labels.get)
selection_rows = [r for r in rows if r.split in ('train', 'val')]
extraction_started = time.time()
features = extract(args, selection_rows, device)
provenance['feature_extraction_seconds'] = time.time() - extraction_started
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))
splits = np.array([r.split for r in selection_rows])
y = np.array([labels[r.label] for r in selection_rows])
mu = features[splits == 'train'].astype(np.float32).mean(0)
sd = features[splits == 'train'].astype(np.float32).std(0) + 1e-6

def tensors(split):
    x = (features[splits == split].astype(np.float32) - mu) / sd
    return torch.from_numpy(x).to(device), torch.from_numpy(y[splits == split]).to(device)
xtr, ytr = tensors('train')
xva, yva = tensors('val')
print('Train features:', tuple(xtr.shape), 'Validation features:', tuple(xva.shape))


In [ ]:
grid_results = []
best = None
started = time.time()
for lr, wd in GRID:
    head = fit_linear(xtr, ytr, len(classes), lr, wd, EPOCHS, SEED, device)
    with torch.no_grad():
        val = summarize(head(xva).float().cpu().numpy(), yva.cpu().numpy(), classes)
    record = {'lr': lr, 'weight_decay': wd, 'seed': SEED, 'epochs': EPOCHS,
              'val_top1': val['top1'], 'val_macro_f1': val['macro_f1'],
              'val_weighted_f1': val['weighted_f1'], 'val_balanced_accuracy': val['balanced_accuracy']}
    grid_results.append(record)
    with (RESULTS / 'grid.jsonl').open('a') as log:
        log.write(json.dumps(record) + '\n')
    print(record, flush=True)
    if best is None or val['top1'] > best['val_top1']:
        best = record
        torch.save({'head': {k: v.detach().cpu().clone() for k, v in head.state_dict().items()},
                    'mean': mu, 'std': sd, 'classes': classes, 'config': vars(args),
                    'best': best, 'base_commit': BASE_COMMIT,
                    'split_sha256': provenance['split_sha256']}, RESULTS / 'best_linear_head.pt')
        (RESULTS / 'validation_metrics.json').write_text(json.dumps(val, indent=2))
(RESULTS / 'grid_summary.json').write_text(json.dumps(grid_results, indent=2))
provenance.update({'seed': SEED, 'epochs': EPOCHS, 'grid': GRID,
                   'training_seconds': time.time() - started, 'best': best})
(RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))
print('Selected on validation top-1:', best)
print('Validation performance only; the test split has not been evaluated.')


## Optional final test evaluation — disabled

Use this only after model selection and the experiment plan are frozen. Leave it disabled during iterative tuning. The head checkpoint uses pickle-based `torch.load`; load only this notebook's own saved checkpoint.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    checkpoint = torch.load(RESULTS / 'best_linear_head.pt', map_location='cpu', weights_only=False)
    test_rows = [r for r in rows if r.split == 'test']
    test_features = extract(args, test_rows, device)
    xte = torch.from_numpy((test_features.astype(np.float32) - checkpoint['mean']) / checkpoint['std']).to(device)
    yte = np.array([labels[r.label] for r in test_rows])
    final_head = torch.nn.Linear(xte.shape[1], len(classes)).to(device)
    final_head.load_state_dict(checkpoint['head'])
    final_head.eval()
    with torch.no_grad():
        test = summarize(final_head(xte).float().cpu().numpy(), yte, classes)
    (RESULTS / 'test_metrics.json').write_text(json.dumps(test, indent=2))
    provenance['test_evaluated'] = True
    (RESULTS / 'provenance.json').write_text(json.dumps(provenance, indent=2))
    print({k: test[k] for k in ['top1', 'macro_f1', 'weighted_f1', 'micro_f1', 'balanced_accuracy']})
else:
    print('Test evaluation skipped.')


In [ ]:
archive = shutil.make_archive('/kaggle/working/texpilot_results', 'zip', RESULTS)
print('Download this results archive:', archive)
from IPython.display import FileLink, display
display(FileLink(archive))
